# Customer Base Table Construction

This notebook transforms the raw Olist transactional data into a customer-level analytical dataset.  
The goal is to create one row per real customer, combining purchase history, spending behavior, delivery experience, and review information into a single table.  

This customer base table will serve as the foundation for the next stages of the project, including RFM segmentation, churn modeling, customer lifetime value estimation, and retention strategy design.

This cell imports the core libraries required for building the customer base table.  
Pandas is used for data manipulation, NumPy supports numerical operations, and Path helps manage file paths cleanly within the project structure.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

This cell loads the main raw datasets required to build the customer-level table.  
The selected files contain customer identifiers, order history, order-level item values, and customer review information.  

These tables provide the key inputs needed to calculate customer activity, monetary value, delivery-related features, and satisfaction signals.

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")

print("Loaded:", 
      "customers", customers.shape,
      "| orders", orders.shape,
      "| order_items", order_items.shape,
      "| reviews", reviews.shape)

This cell converts the relevant date columns in the orders and reviews datasets into proper datetime format.  
This is necessary because later features such as recency, customer age, delivery duration, and delivery delay all depend on valid timestamps.  

The final line also defines the reference date for the analysis by taking the latest purchase date in the dataset. This date is used as the anchor point for recency calculations.

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in order_date_cols:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

review_date_cols = ["review_creation_date", "review_answer_timestamp"]
for c in review_date_cols:
    if c in reviews.columns:
        reviews[c] = pd.to_datetime(reviews[c], errors="coerce")

AS_OF_DATE = orders["order_purchase_timestamp"].max()
AS_OF_DATE

This cell creates a mapping between `customer_id` and `customer_unique_id`.  
The purpose is to distinguish between the system-level customer identifier and the real-world unique customer identifier.  

This mapping is important because the final customer base table should represent one row per actual customer, not one row per system-generated customer record.

In [ ]:
cust_map = customers[["customer_id", "customer_unique_id"]].drop_duplicates()
cust_map.shape, cust_map.head()

This cell examines the distribution of order statuses and then filters the dataset to include only delivered orders.  
This ensures that the customer base table is built using completed transactions that reflect real purchases and completed customer experiences.  

Orders that were cancelled or not fulfilled are excluded because they do not represent successful transactions for retention and value analysis.

In [ ]:
orders["order_status"].value_counts()
orders_delivered = orders[orders["order_status"] == "delivered"].copy()
orders_delivered.shape

This cell aggregates the order items table to the order level and calculates monetary features for each order.  
It computes the total item value, total freight value, total order value, number of items, number of unique products, and number of unique sellers.  

This step is necessary because the raw order items table is stored at item-level grain, while the customer base table requires order-level summaries before aggregation to the customer level.

In [ ]:
order_value = (
    order_items.assign(order_total=order_items["price"] + order_items["freight_value"])
    .groupby("order_id", as_index=False)
    .agg(
        items_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        order_total=("order_total", "sum"),
        n_items=("order_item_id", "count"),
        n_unique_products=("product_id", "nunique"),
        n_unique_sellers=("seller_id", "nunique"),
    )
)

order_value.head(), order_value.shape

This cell creates delivery-related features for each delivered order.  
It calculates the total delivery time in days and the delivery delay in days compared with the estimated delivery date.  

These features are useful because delivery experience can influence customer satisfaction, repeat purchase behaviour, and churn risk.

In [ ]:
delivery = orders_delivered[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]].copy()

delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"] - delivery["order_purchase_timestamp"]
).dt.days

delivery["delivery_delay_days"] = (
    delivery["order_delivered_customer_date"] - delivery["order_estimated_delivery_date"]
).dt.days

delivery[["delivery_days", "delivery_delay_days"]].describe()

This cell aggregates review information to the order level.  
It calculates the average review score for each order and creates a binary flag indicating whether the order received at least one review.  

These features act as a proxy for customer satisfaction and engagement, which can later be used in segmentation and churn analysis.

In [ ]:
order_review = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        has_review=("review_score", lambda s: int(s.notna().any())),
    )
)

order_review.head(), order_review.shape

This cell combines the delivered orders table with the customer mapping, order value features, delivery features, and review features to create a unified order-level master table.  
The result is one enriched table where each row represents a delivered order along with its associated customer, monetary, delivery, and review information.  

This order-level master table is the immediate input for building the final customer-level dataset.

In [ ]:
order_master = (
    orders_delivered[["order_id", "customer_id", "order_purchase_timestamp"]]
    .merge(cust_map, on="customer_id", how="left")
    .merge(order_value, on="order_id", how="left")
    .merge(delivery[["order_id", "delivery_days", "delivery_delay_days"]], on="order_id", how="left")
    .merge(order_review, on="order_id", how="left")
)

order_master.head(), order_master.shape

This cell checks whether any delivered orders are missing a `customer_unique_id` after merging with the customer mapping table.  
A missing rate close to zero confirms that the join was successful and that customer identity has been preserved correctly.  

This is an important validation step before aggregating the data to the customer level.

In [ ]:
order_master["customer_unique_id"].isna().mean()

This cell aggregates the order-level master table to create the final customer-level base table.  
Each row now represents one unique customer, with features summarising their purchase history, spending behaviour, delivery experience, and review activity.  

Additional customer-level time features are also created, including recency and customer age, which will later be used in segmentation, churn modelling, and value analysis.

In [ ]:
customer_base = (
    order_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        n_orders=("order_id", "nunique"),
        total_spend=("order_total", "sum"),
        avg_order_value=("order_total", "mean"),
        avg_items=("n_items", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        review_rate=("has_review", "mean"),  # share of orders reviewed
    )
)

customer_base["recency_days"] = (AS_OF_DATE - customer_base["last_purchase"]).dt.days
customer_base["customer_age_days"] = (AS_OF_DATE - customer_base["first_purchase"]).dt.days

customer_base.head(), customer_base.shape

This cell performs basic validation checks on the customer base table.  
It confirms that each customer appears only once, verifies that recency values are non-negative, and provides summary statistics for key variables such as number of orders, total spend, and recency.  

These checks help ensure that the aggregation logic is correct and that the resulting dataset is suitable for downstream analysis.

In [ ]:
# Grain: one row per real customer
assert customer_base["customer_unique_id"].is_unique

# No negative recency
assert (customer_base["recency_days"] >= 0).all()

customer_base[["n_orders", "total_spend", "recency_days"]].describe()

This cell saves the customer base table as a processed CSV file in the project’s processed data directory.  
Saving the output makes the workflow reproducible and allows later notebooks to load the cleaned customer-level dataset directly instead of rebuilding it from raw data each time.

In [ ]:
out_path = PROCESSED_DIR / "customer_base.csv"
customer_base.to_csv(out_path, index=False)
print("Saved:", out_path)

This cell prints a small set of summary outputs for the customer base table.  
It shows the overall dataset shape, the distribution of order counts per customer, and the distribution of recency values.  

These outputs provide a final sanity check and help us understand the behavioural profile of customers before moving into RFM segmentation.

In [ ]:
print("Customer base shape:")
print(customer_base.shape)

print("\nTop 5 most common n_orders values:")
print(customer_base["n_orders"].value_counts().head(5))

print("\nRecency (days) distribution:")
print(customer_base["recency_days"].describe())